# Notebook 1 - Build the study area and explore the Lower Moulouya

<a target="_blank" href="https://colab.research.google.com/github/khouakhi/UMP_EO_training/blob/main/notebooks/01_aoi_explore_lower_moulouya.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


## Big picture

We study **northeastern Morocco**, where the **Lower Moulouya** river reaches the Mediterranean. The **2025–2026** rainy period was unusually wet. Later notebooks compare the **extended winter–spring wet season (December-April)**, labelled by the **April** that closes each window (e.g. wet season **2025/26** ends April 2026), using [**CHIRPS**](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY), [**Sentinel-2**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), and related layers.

## Main question

> How did the unusually wet **2025/26** wet season affect **surface water** and **vegetation** in the Lower Moulouya compared with the **2024/25** wet season and a **long-term baseline of wet seasons ending April 2017-April 2024**?

## Objective in *this* notebook

- Get an **AOI** (area of interest), which is the hydrological catchment boundary of the Lower Moulouya from HydroSHEDS.
- How to use **geemap** with a **Google Satellite** basemap for geographic context.
- How to split [**Copernicus DEM GLO-30**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30) (the data) from **hillshade** (the visualisation) into two clear parts.


**Time tip (3 h module):** spend about **25-30 minutes** here, including discussion with your group.


In [1]:
# Install packages (Colab often needs a fresh install each session)
# !pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# If you run this notebook locally, run `ee.Authenticate()` once before `ee.Initialize`.
# If the Colab pop-up fails, try: ee.Authenticate(auth_mode="colab")
# Mapping notebooks use `Map.add_basemap("SATELLITE")` so you always have photo context under EE layers.


In [2]:
# Connect to Google Earth Engine using your cloud project ID.
# Set this to your own Google Earth Engine cloud project ID before running.
EE_PROJECT = "YOUR_GEE_PROJECT_ID"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialised with project:", EE_PROJECT)


Earth Engine initialised with project: ee-gee-hydro


## Define the AOI - HydroSHEDS hydrological unit

Here, we load a **pre-defined basin polygon** from **HydroSHEDS** (WWF). This is a **level-08** sub-basin: one level in a global hierarchy of nested catchments. Everyone uses the same **`HYBAS_ID`**, so your maps match other notebooks and published basin codes.

This notebook only defines the **basin** (no rainfall code here; [**CHIRPS Daily**](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY) starts in notebook **02**).

**Target basin - Moulouya (Lower Moulouya in HydroBASINS terms):** `HYBAS_ID = 1080030220` on [**HydroSHEDS hybas level 8**](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_v1_Basins_hybas_8) (`WWF/HydroSHEDS/v1/Basins/hybas_8`).

Why hydrological units?

- The boundary follows **drainage** (ridges and outlets), not an arbitrary rectangle.
- The same **`HYBAS_ID`** is easy to **look up**, cite, and **re-use** in the next notebook (clip, zonal stats, exports).

Run the code cell below, then check the **feature count** prints **1**. If it prints **0**, the filter or dataset path is wrong; ask your instructor.


In [3]:
# Study area: Moulouya basin - HydroSHEDS level-8 hydrological unit (WWF)
# Dataset: WWF/HydroSHEDS/v1/Basins/hybas_8 - use the same HYBAS_ID in every notebook for consistency.

HYBAS_ID = 1080030220

MOULOUYA_BASIN_H08 = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8").filter(
    ee.Filter.eq("HYBAS_ID", HYBAS_ID)
)

# Geometry used for clips, filterBounds, reduceRegion, etc.
LOWER_MOULOUYA_AOI = MOULOUYA_BASIN_H08.geometry()


In [ ]:
print("HydroSHEDS hybas_8 features with HYBAS_ID =", HYBAS_ID, ":", MOULOUYA_BASIN_H08.size().getInfo())


## 1. Build the Copernicus DEM

Here we create one **elevation image** over the basin: `DEM_GLO30` (metres, [**Copernicus DEM GLO-30**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30)). This cell is **only** about loading data and fixing the **native projection** after `mosaic()`, a requirement for sensible `Terrain` outputs (see the catalogue **Description** tab for caveats).

The **next** section adds the **map** (satellite basemap + hillshade). Keeping **data** and **visualisation** apart is good practice and easier to debug.


In [4]:
glo30 = ee.ImageCollection("COPERNICUS/DEM/GLO30")
native_proj = glo30.first().projection()
DEM_GLO30 = (
    glo30.select("DEM")
    .mosaic()
    .setDefaultProjection(native_proj)
    .rename("elevation_m")
    .clip(LOWER_MOULOUYA_AOI)
)
print("DEM ready: Copernicus GLO-30, one band elevation_m, clipped to the basin.")


DEM ready: Copernicus GLO-30, one band elevation_m, clipped to the basin.


## 2. Visualise on a **satellite** basemap

**Basemap:** `Map.add_basemap("SATELLITE")` adds **Google Satellite** imagery (true colour context; not an Earth Engine `Image`).

**Overlay:** **Hillshade** is computed from `DEM_GLO30` for **display only**. Multiplying elevations by **20** before `Terrain.hillshade` follows the [**DEM GLO-30**](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30) catalogue recipe so gentle relief shows up; it is **not** a physical change to the DEM.

**On top:** the [**HydroSHEDS**](https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroSHEDS_v1_Basins_hybas_8) basin outline in bright yellow so you can match drainage limits to fields and settlements.

Use the layer control to turn **DEM** on or off (hidden by default so the tutorial stays uncluttered).

**Tip:** if the map does not appear in Colab, run the cell again or click "Show map" if Colab collapses the widget.


In [6]:
# Layer and style definitions.
basin_vis = {"fillColor": "00000000", "color": "ffff00", "width": 3}
basin_layer_vis = {}
hillshade_layer = ee.Terrain.hillshade(DEM_GLO30.multiply(20.0))
hillshade_vis = {"min": 0, "max": 255}
dem_vis = {"min": 0, "max": 2500, "palette": ["232359", "1d91c0", "8ed368", "fcfdb5"]}

# Mapping calls grouped together.
Map = geemap.Map()
Map.add_basemap("SATELLITE")
Map.centerObject(LOWER_MOULOUYA_AOI, 9)
Map.addLayer(hillshade_layer, hillshade_vis, "Hillshade", opacity=0.55)
Map.addLayer(MOULOUYA_BASIN_H08.style(**basin_vis), basin_layer_vis, "Basin")
Map.addLayer(DEM_GLO30, dem_vis, "DEM m", shown=False)

Map


Map(center=[34.88374700771184, -2.5725774983515297], controls=(WidgetControl(options=['position', 'transparent…

## Export the AOI as an Earth Engine asset (optional)

Only run this if your instructor asks you to save the AOI to your own Earth Engine cloud project. You need **write** permission on that project.

Otherwise, every notebook can rebuild the same geometry from code (as we do here).


In [ ]:
# OPTIONAL - uncomment only if you have write access and want a saved asset (attributes preserved)
# task = ee.batch.Export.table.toAsset(
#     collection=MOULOUYA_BASIN_H08,
#     description="moulouya_hybas_h08_1080030220",
#     assetId="projects/YOUR_GEE_PROJECT_ID/assets/MOULOUYA_BASIN_H08",
# )
# task.start()
# print("Export started - check Tasks tab in the Earth Engine Code Editor")
print("Skipping export by default.")


## Short written task (5 minutes)

In your own words, list **three land-cover / land-use** types you expect to see inside the AOI (for example: irrigated fields, urban, bare soil). Say **where** in the AOI each type is most likely.

**Next notebook:** `02_rainfall_chirps_anomaly.ipynb`.
